In [ ]:
from pathlib import Path

import numpy as np 
import pandas as pd
import pymeshlab
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plyfile import PlyData
from sklearn.neighbors import NearestNeighbors

## Pheno4D

In [ ]:
data_folder = Path('../data/Pheno4D/raw/Maize01')
file_paths = sorted(data_folder.glob('*'))

In [ ]:
df = pd.read_csv(file_paths[0], sep=' ', header=None, index_col=False)

In [ ]:
assert len(df.columns) >= 3, f"Expected at least 3 columns, got {len(df.columns)}"

In [ ]:
data = np.loadtxt(file_paths[0])
data

In [ ]:
SCALAR_DIM = 4
points = data[:, :3]
labels = data[:, SCALAR_DIM] 

In [ ]:
unique_labels = np.unique(labels)
color_map = plt.get_cmap("tab20")  # or any other matplotlib colormap
label_to_color = {label: color_map(i / len(unique_labels))[:3] for i, label in enumerate(unique_labels)}
colors = np.array([label_to_color[label] for label in labels])

In [ ]:
fraction = 0.1
N = points.shape[0]
sample_indices = np.random.choice(N, size=int(N * fraction), replace=False)

In [ ]:
labels

### Get one leaf

In [ ]:
sample_indices = np.where(points[:, 2] > -5)[0]
sample_indices = np.where(labels[:] == 1.)
sample_indices

In [ ]:
points_downsampled = points[sample_indices]
labels_downdownsampled = labels[sample_indices]
colors_downsampled = colors[sample_indices]

In [ ]:
colors.shape

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points_downsampled[:, 0], points_downsampled[:, 1], points_downsampled[:, 2], c=colors_downsampled, s=1)  # s=1 for small point size

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Point Cloud Colored by Column')

plt.show()

In [ ]:
plt.hist(points[:, 2], bins=100)
plt.xlim(-10, 1)

In [ ]:
np.median(points[:, 2])

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=points_downsampled[:, 0],
    y=points_downsampled[:, 1],
    z=points_downsampled[:, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=colors_downsampled,  # Can use labels or other scalar for color
        colorscale='Viridis',
        opacity=0.8
    )
)])
fig.show()

In [ ]:
# # Create Open3D point cloud
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(points_downsampled)

# # If you have color data (float RGB values between 0 and 1)
# pcd.colors = o3d.utility.Vector3dVector(colors_downsampled)

# # Save to PLY
# o3d.io.write_point_cloud("../downsampled_point_cloud.ply", pcd)

## BonnBeetClouds3D

In [ ]:
base_path = Path('~/public-mnt/Evaluation/Projects/KP0034_jgajardo/data/external/BonnBeetClouds3Dv0').expanduser().absolute()
base_path.exists()

In [ ]:
data_folder = base_path / 'val'
sample_file = list(data_folder.glob('*'))[0]

In [ ]:
open3d_pcd = o3d.io.read_point_cloud(str(sample_file))
pcd_np = np.asarray(open3d_pcd.points)

Open3D doesn't support reading scalar fields from PLY files, so we'll use `plyfile` instead.

Iterate over folder to check the amount of scalar fields (look for leaf tip)

In [ ]:
def get_ply_attributes(ply_file_path):
    ply_data = PlyData.read(ply_file_path)
    vertex = ply_data['vertex']
    return vertex.properties

In [ ]:
get_ply_attributes(sample_file)

In [ ]:
for file in data_folder.glob('*'):
    print(len(get_ply_attributes(sample_file)))

There is no keypoint_id in the data as advertised in the paper :(

## TrackPlant3D

In [ ]:
# Helper functions
def load_point_cloud(file_path):
    data = np.loadtxt(file_path)
    return data[:, :3], data[:, 3:]  # Assuming last columns are colors

def plot_interactive_with_keypoint(points, keypoint_index):
    """
    Interactive plot with keypoint highlighted
    """
    colors = ['gray'] * len(points)
    sizes = [2] * len(points)

    # Highlight keypoint
    keypoint_pos = points[keypoint_index]
    colors[keypoint_index] = 'red'
    sizes[keypoint_index] = 8
    
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=sizes,
            color=colors,
            opacity=0.8
        ),
        text=[f'Index: {idx}' for idx in range(points.shape[0])],  # Hover info
        hovertemplate='Index: %{text}<br>X: %{x}<br>Y: %{y}<br>Z: %{z}<extra></extra>'
    )])
    
    fig.update_layout(
        title=f'Point Cloud with Keypoint (Index: {keypoint_index})',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        )
    )
    fig.show()

### EDA

In [ ]:
crop = 'tomato'
data_folder = Path(f'../data/TrackPlant3D/gt/{crop}')
file_paths = sorted(data_folder.glob('*'), key=lambda x: int(x.stem.split('_')[0]))

In [ ]:
data, labels = load_point_cloud(file_paths[0])
data.shape, labels.shape

In [ ]:
number_of_leaves = []
for file_path in file_paths:
    labels = load_point_cloud(file_path)[1]
    #print(f"{file_path.name}: {np.unique(labels)}")
    number_of_leaves.append(len(np.unique(labels)) - 1)
print(f"Average number of leaves in {crop}: {np.sum(number_of_leaves)}")

### Leaf tips labels (output of LabelCloud)

In [ ]:
bin_files = sorted(Path('../data/TrackPlant3D/keypoints/maize/arrays').glob('*.bin'), key=lambda x: int(x.stem.split('_')[0]))
path = bin_files[-1]
print(path)
labels = np.fromfile(path, dtype=np.int8)
np.unique(labels, return_counts=True)

### Export single leaf for meshing

In [ ]:
crop = 'maize'
data_folder = Path(f'../data/TrackPlant3D/gt/{crop}')
file_paths = sorted(data_folder.glob('*'), key=lambda x: int(x.stem.split('_')[0]))

In [ ]:
file_path = file_paths[0]
points, labels = load_point_cloud(file_path)
file_path.name, points.shape, labels.shape

In [ ]:
unique_labels = np.unique(labels)
unique_labels

In [ ]:
leaf_id = 1.0 # start from 1.0 for the first leaf
assert leaf_id in unique_labels, f"Leaf ID {leaf_id} not found in unique labels: {unique_labels}"
assert leaf_id != 0.0, "Leaf ID should not be 0.0, which represents the stem."

sample_indices = np.where(labels[:] == unique_labels[int(leaf_id)])[0]
leaf_points = points[sample_indices]
leaf_points.shape

In [ ]:
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(leaf_points)
# o3d.io.write_point_cloud(f"../{file_path.stem}_leaf{int(leaf_id)}.ply", pcd)

Plot keypoint

In [ ]:
index = 5311  # Just for checking, found with Meshlab -> meshing with ball pivot -> Colorize by geodesic distance -> The info to check point index
plot_interactive_with_keypoint(leaf_points, keypoint_index=index)

**Goal:** find keypoint automatically with pymeshlab!

### Explore automatic keypoint detection

In [ ]:
# Mesh the leaf to enable geodesic distance computation
ms = pymeshlab.MeshSet()
mesh = pymeshlab.Mesh(vertex_matrix=leaf_points)
ms.add_mesh(mesh, f"leaf_{int(leaf_id)}")
ms.generate_surface_reconstruction_ball_pivoting()

In [ ]:
centroid = np.mean(leaf_points, axis=0)
leaf_centroid_index = np.argmin(np.linalg.norm(leaf_points - centroid, axis=1)) # closest data point to the centroid
leaf_centroid = np.array(leaf_points[leaf_centroid_index], dtype=np.float32)

In [ ]:
# Plot leaf centroid
plot_interactive_with_keypoint(leaf_points, keypoint_index=leaf_centroid_index)

In [ ]:
ms.compute_scalar_by_geodesic_distance_from_given_point_per_vertex(startpoint=leaf_centroid)
geodist = ms.current_mesh().vertex_scalar_array()
geodist.shape

In [ ]:
tip_index = np.argmax(geodist)
tip_index

In [ ]:
# Plot farthest point from centroid (either tip or insertion point)
plot_interactive_with_keypoint(leaf_points, keypoint_index=tip_index)

But how to know if farthest point from leaf centroid is the tip and insertion point? use stem points!

In [ ]:
stem_indices = np.where(labels == 0)[0]
stem_points = points[stem_indices]
stem_points

In [ ]:
leaf_stem_points = np.vstack((leaf_points, stem_points))

In [ ]:
plot_interactive_with_keypoint(leaf_stem_points, keypoint_index=leaf_centroid_index)

In [ ]:
# Find the closest geodesic point with stem label (in first `len(stem_points)` entries)
ms = pymeshlab.MeshSet()
mesh = pymeshlab.Mesh(vertex_matrix=leaf_stem_points)
ms.add_mesh(mesh, f"leaf_stem_{leaf_id}")
ms.generate_surface_reconstruction_ball_pivoting() # Meshing leaf and stem together doesn't work well though

Another approach is to go first get farthest point from leaf centroid (yielding keypoint A), then get the farthest point from keypoint A along th leaf (yielding keypoint B), then we can find which on is the insertion point by checking which one is closer to a stem point

In [ ]:
# This seems to work nicely!
ms.add_mesh(pymeshlab.Mesh(vertex_matrix=leaf_points), f"leaf_{leaf_id}")
ms.generate_surface_reconstruction_ball_pivoting()

centroid = np.mean(leaf_points, axis=0)
first_index = np.argmin(np.linalg.norm(leaf_points - centroid, axis=1))
first_point = np.array(leaf_points[first_index], dtype=np.float32)

# First get farthest point from centroid (could be tip or insertion point)
ms.compute_scalar_by_geodesic_distance_from_given_point_per_vertex(startpoint=first_point)
geodist1 = ms.current_mesh().vertex_scalar_array()
index_A = np.argmax(geodist1)
point_A = leaf_points[index_A]

# Now get farthest point from point_A
ms.compute_scalar_by_geodesic_distance_from_given_point_per_vertex(startpoint=np.array(point_A, dtype=np.float32))
geodist2 = ms.current_mesh().vertex_scalar_array()
index_B = np.argmax(geodist2)
point_B = leaf_points[index_B]

In [ ]:
keypoint_index = index_A # change to index_B to plot the other point
plot_interactive_with_keypoint(leaf_points, keypoint_index=keypoint_index)

In [ ]:
np.linalg.norm(stem_points - point_A, axis=1).min() # this is smaller than for B so this must be insertion point! True

In [ ]:
np.linalg.norm(stem_points - point_B, axis=1).min()

Point A must be insertion point and point B the tip, because the it has a smaller minimum distance to the stem

More concisely:

In [ ]:
# Get distances and determine insertion point
distances = [np.linalg.norm(stem_points - point_A, axis=1).min(),
            np.linalg.norm(stem_points - point_B, axis=1).min()]
points = [point_A, point_B]
indices = [index_A, index_B]

insertion_idx = np.argmin(distances)
tip_idx = 1 - insertion_idx  # The other point

insertion_point = points[insertion_idx]
insertion_index = indices[insertion_idx]
tip_point = points[tip_idx]
tip_index = indices[tip_idx]

In [ ]:
plot_interactive_with_keypoint(leaf_points, keypoint_index=insertion_index)